In [192]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
folder_path = '/content/drive/MyDrive/Colab Notebooks/Personal Projects/Project_2_sales_analysis/'


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [193]:
import pandas as pd
import numpy as np

np.random.seed(42)

# -----------------------------
# 1. CUSTOMERS
# -----------------------------

n_customers = 2500

cities = {
    "Hyderabad": "Telangana",
    "Bengaluru": "Karnataka",
    "Mumbai": "Maharashtra",
    "Pune": "Maharashtra",
    "Delhi": "Delhi",
    "Chennai": "Tamil Nadu",
    "Kolkata": "West Bengal",
    "Ahmedabad": "Gujarat",
    "Jaipur": "Rajasthan",
    "Kochi": "Kerala"
}

city_list = list(cities.keys())

customers = pd.DataFrame({
    "customer_id": [f"C{str(i).zfill(5)}" for i in range(1, n_customers + 1)],
    "customer_name": [f"Customer_{i}" for i in range(1, n_customers + 1)],
    "customer_segment": np.random.choice(
        ["Consumer", "Small Business", "Enterprise"],
        size=n_customers,
        p=[0.60, 0.30, 0.10]
    ),
    "city": np.random.choice(city_list, size=n_customers),
})

customers["state"] = customers["city"].map(cities)

customers["customer_since"] = pd.to_datetime(
    np.random.choice(
        pd.date_range("2020-01-01", "2025-12-31"),
        size=n_customers
    )
)

# -----------------------------
# 2. PRODUCTS
# -----------------------------

products_data = []

categories = {
    "Electronics": ["Laptops", "Mobile Phones", "Accessories"],
    "Furniture": ["Chairs", "Desks", "Storage"],
    "Home Appliances": ["Kitchen", "Cleaning", "Climate"],
    "Office Supplies": ["Stationery", "Paper", "Writing"]
}

product_id = 1

for category, subcategories in categories.items():
    for subcategory in subcategories:
        for i in range(1, 7):

            base_price = {
                "Electronics": np.random.uniform(15000, 80000),
                "Furniture": np.random.uniform(3000, 30000),
                "Home Appliances": np.random.uniform(2000, 25000),
                "Office Supplies": np.random.uniform(200, 5000)
            }[category]

            products_data.append({
                "product_id": f"P{str(product_id).zfill(4)}",
                "product_name": f"{subcategory} Product {i}",
                "category": category,
                "subcategory": subcategory,
                "unit_cost": round(base_price * np.random.uniform(0.55, 0.80), 2),
                "base_price": round(base_price, 2)
            })

            product_id += 1

products = pd.DataFrame(products_data)

# -----------------------------
# 3. ORDERS
# -----------------------------

n_orders = 30000

orders = pd.DataFrame({
    "order_id": [f"O{str(i).zfill(6)}" for i in range(1, n_orders + 1)],
    "order_date": np.random.choice(
        pd.date_range("2023-01-01", "2025-12-31"),
        size=n_orders
    ),
    "customer_id": np.random.choice(
        customers["customer_id"],
        size=n_orders
    ),
    "product_id": np.random.choice(
        products["product_id"],
        size=n_orders
    ),
    "quantity": np.random.choice(
        [1, 2, 3, 4, 5],
        size=n_orders,
        p=[0.50, 0.25, 0.12, 0.08, 0.05]
    ),
    "discount_pct": np.random.choice(
        [0, 0.05, 0.10, 0.15, 0.20, 0.25],
        size=n_orders,
        p=[0.20, 0.25, 0.25, 0.15, 0.10, 0.05]
    ),
    "shipping_cost": np.random.uniform(50, 1500, size=n_orders),
    "sales_channel": np.random.choice(
        ["Online", "Store"],
        size=n_orders,
        p=[0.70, 0.30]
    ),
    "order_status": np.random.choice(
        ["Completed", "Returned", "Cancelled"],
        size=n_orders,
        p=[0.90, 0.07, 0.03]
    )
})

orders["order_date"] = pd.to_datetime(orders["order_date"])

# Join product pricing information
orders = orders.merge(
    products[["product_id", "base_price", "unit_cost"]],
    on="product_id",
    how="left"
)

# Calculate selling price after discount
orders["unit_price"] = (
    orders["base_price"] *
    (1 - orders["discount_pct"])
).round(2)

# Revenue
orders["revenue"] = (
    orders["quantity"] *
    orders["unit_price"]
).round(2)

# Cost
orders["product_cost"] = (
    orders["quantity"] *
    orders["unit_cost"]
).round(2)

# Profit
orders["profit"] = (
    orders["revenue"] -
    orders["product_cost"] -
    orders["shipping_cost"]
).round(2)

# -----------------------------
# 4. INTRODUCE DATA QUALITY ISSUES
# -----------------------------

# Missing customer city for a small number of customers
missing_customer_idx = np.random.choice(
    customers.index,
    size=30,
    replace=False
)

customers.loc[missing_customer_idx, "city"] = np.nan

# Missing product costs
missing_product_idx = np.random.choice(
    products.index,
    size=4,
    replace=False
)

products.loc[missing_product_idx, "unit_cost"] = np.nan

# A few unusual quantities
outlier_idx = np.random.choice(
    orders.index,
    size=15,
    replace=False
)

orders.loc[outlier_idx, "quantity"] = np.random.choice(
    [20, 25, 50],
    size=len(outlier_idx)
)

# A few missing discounts
missing_discount_idx = np.random.choice(
    orders.index,
    size=50,
    replace=False
)

orders.loc[missing_discount_idx, "discount_pct"] = np.nan

# A few duplicate-looking rows
duplicate_rows = orders.sample(10, random_state=42)

orders = pd.concat(
    [orders, duplicate_rows],
    ignore_index=True
)

# -----------------------------
# 5. SAVE RAW DATA
# -----------------------------

customers.to_csv(
    folder_path+"customers.csv",
    index=False
)

products.to_csv(
    folder_path+"products.csv",
    index=False
)

orders.to_csv(
    folder_path+"orders.csv",
    index=False
)

print("Dataset created successfully!")
print(f"Customers: {len(customers):,}")
print(f"Products: {len(products):,}")
print(f"Orders: {len(orders):,}")

Dataset created successfully!
Customers: 2,500
Products: 72
Orders: 30,010


# Data Profiling

In [194]:
#Checking the data - customers
print(customers.shape)
customers.head(5)


(2500, 6)


,customer_id,customer_name,customer_segment,city,state,customer_since
0,C00001,Customer_1,Consumer,Mumbai,Maharashtra,2025-03-02
1,C00002,Customer_2,Enterprise,Pune,Maharashtra,2025-09-26
2,C00003,Customer_3,Small Business,Kolkata,West Bengal,2024-02-23
3,C00004,Customer_4,Consumer,Mumbai,Maharashtra,2022-09-28
4,C00005,Customer_5,Consumer,Jaipur,Rajasthan,2022-10-30


In [195]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   customer_id       2500 non-null   object        
 1   customer_name     2500 non-null   object        
 2   customer_segment  2500 non-null   object        
 3   city              2470 non-null   object        
 4   state             2500 non-null   object        
 5   customer_since    2500 non-null   datetime64[ns]
dtypes: datetime64[ns](1), object(5)
memory usage: 117.3+ KB


In [196]:
customers[customers["city"].isna()]

,customer_id,customer_name,customer_segment,city,state,customer_since
55,C00056,Customer_56,Enterprise,NaN,Gujarat,2022-02-26
129,C00130,Customer_130,Consumer,NaN,Kerala,2025-07-17
182,C00183,Customer_183,Enterprise,NaN,Kerala,2022-07-27
219,C00220,Customer_220,Small Business,NaN,Tamil Nadu,2025-08-22
350,C00351,Customer_351,Consumer,NaN,Maharashtra,2020-10-10
415,C00416,Customer_416,Consumer,NaN,Gujarat,2021-07-20
461,C00462,Customer_462,Small Business,NaN,Maharashtra,2023-04-21
498,C00499,Customer_499,Enterprise,NaN,Maharashtra,2021-06-30
555,C00556,Customer_556,Consumer,NaN,Telangana,2022-07-08
766,C00767,Customer_767,Small Business,NaN,Tamil Nadu,2024-09-25


In [197]:
customers[customers["city"].isna()]["state"].value_counts(dropna=False)

,count
state,
Maharashtra,12
Tamil Nadu,6
Gujarat,4
Kerala,2
Telangana,2
Karnataka,2
West Bengal,2


In [198]:
customers["customer_segment"].value_counts(dropna=False)

,count
customer_segment,
Consumer,1486
Small Business,769
Enterprise,245


In [199]:
#customers - Basic integrity checks
customers["customer_id"].nunique()

2500

In [200]:
customers["customer_id"].duplicated().sum()

np.int64(0)

In [201]:
customers["customer_since"].min(), customers["customer_since"].max()

(Timestamp('2020-01-02 00:00:00'), Timestamp('2025-12-31 00:00:00'))

In [202]:
#Checking the data - products

products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72 entries, 0 to 71
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    72 non-null     object 
 1   product_name  72 non-null     object 
 2   category      72 non-null     object 
 3   subcategory   72 non-null     object 
 4   unit_cost     68 non-null     float64
 5   base_price    72 non-null     float64
dtypes: float64(2), object(4)
memory usage: 3.5+ KB


In [203]:
products.shape

(72, 6)

In [204]:
products.head()

,product_id,product_name,category,subcategory,unit_cost,base_price
0,P0001,Laptops Product 1,Electronics,Laptops,44819.33,76307.73
1,P0002,Laptops Product 2,Electronics,Laptops,31099.07,48149.03
2,P0003,Laptops Product 3,Electronics,Laptops,29587.37,46962.97
3,P0004,Laptops Product 4,Electronics,Laptops,30150.61,42415.33
4,P0005,Laptops Product 5,Electronics,Laptops,24460.10,32063.17


In [205]:
products["product_id"].nunique()

72

In [206]:
products["product_id"].duplicated().sum()

np.int64(0)

In [207]:
products.isna().sum()

,0
product_id,0
product_name,0
category,0
subcategory,0
unit_cost,4
base_price,0


base_price = normal customer selling price before discount

unit_price = actual customer selling price after discount

unit_cost = company's cost

In [208]:
products[products["unit_cost"] > products["base_price"]]

,product_id,product_name,category,subcategory,unit_cost,base_price


In [209]:
products["category"].value_counts()

,count
category,
Electronics,18
Furniture,18
Home Appliances,18
Office Supplies,18


In [210]:
products["subcategory"].value_counts()

,count
subcategory,
Laptops,6
Mobile Phones,6
Accessories,6
Chairs,6
Desks,6
Storage,6
Kitchen,6
Cleaning,6
Climate,6


In [211]:
pd.crosstab(
    products["category"],
    products["subcategory"]
)

subcategory,Accessories,Chairs,Cleaning,Climate,Desks,Kitchen,Laptops,Mobile Phones,Paper,Stationery,Storage,Writing
category,,,,,,,,,,,,
Electronics,6,0,0,0,0,0,6,6,0,0,0,0
Furniture,0,6,0,0,6,0,0,0,0,0,6,0
Home Appliances,0,0,6,6,0,6,0,0,0,0,0,0
Office Supplies,0,0,0,0,0,0,0,0,6,6,0,6


In [212]:
products[products["unit_cost"].isna()]

,product_id,product_name,category,subcategory,unit_cost,base_price
10,P0011,Mobile Phones Product 5,Electronics,Mobile Phones,NaN,76595.54
15,P0016,Accessories Product 4,Electronics,Accessories,NaN,77138.70
31,P0032,Storage Product 2,Furniture,Storage,NaN,24167.24
69,P0070,Writing Product 4,Office Supplies,Writing,NaN,421.13


In [213]:
#Checking the data - orders
orders.shape

(30010, 15)

In [214]:
orders.head()

,order_id,order_date,customer_id,product_id,quantity,discount_pct,shipping_cost,sales_channel,order_status,base_price,unit_cost,unit_price,revenue,product_cost,profit
0,O000001,2025-09-24,C00906,P0071,1,0.05,281.576555,Online,Completed,4370.25,3126.80,4151.74,4151.74,3126.80,743.36
1,O000002,2025-01-03,C00798,P0025,1,0.05,169.343988,Online,Completed,13173.89,8040.17,12515.20,12515.20,8040.17,4305.69
2,O000003,2023-04-01,C02034,P0049,1,0.05,85.972542,Online,Completed,3620.60,2285.05,3439.57,3439.57,2285.05,1068.55
3,O000004,2023-06-16,C01587,P0063,3,0.00,1393.136007,Store,Completed,3877.55,3036.56,3877.55,11632.65,9109.68,1129.83
4,O000005,2025-08-16,C01778,P0027,1,0.10,1036.538451,Online,Completed,26372.42,16854.64,23735.18,23735.18,16854.64,5844.00


Column	Meaning

base_price= Selling price before discount

discount_pct=	Discount offered

unit_price=	Actual selling price after discount

unit_cost=	Company's cost per unit

quantity=	Units purchased

revenue=	quantity × unit_price

product_cost=	quantity × unit_cost

profit=	revenue - product_cost - shipping_cost

In [215]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30010 entries, 0 to 30009
Data columns (total 15 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   order_id       30010 non-null  object        
 1   order_date     30010 non-null  datetime64[ns]
 2   customer_id    30010 non-null  object        
 3   product_id     30010 non-null  object        
 4   quantity       30010 non-null  int64         
 5   discount_pct   29960 non-null  float64       
 6   shipping_cost  30010 non-null  float64       
 7   sales_channel  30010 non-null  object        
 8   order_status   30010 non-null  object        
 9   base_price     30010 non-null  float64       
 10  unit_cost      30010 non-null  float64       
 11  unit_price     30010 non-null  float64       
 12  revenue        30010 non-null  float64       
 13  product_cost   30010 non-null  float64       
 14  profit         30010 non-null  float64       
dtypes: datetime64[ns](1

In [216]:
orders.isna().sum()

,0
order_id,0
order_date,0
customer_id,0
product_id,0
quantity,0
discount_pct,50
shipping_cost,0
sales_channel,0
order_status,0
base_price,0


In [217]:
orders["sales_channel"].value_counts()

,count
sales_channel,
Online,21105
Store,8905


In [218]:
orders["order_status"].value_counts()

,count
order_status,
Completed,27017
Returned,2102
Cancelled,891


In [219]:
#Duplicate orders
orders["order_id"].nunique()

30000

In [220]:
orders["order_id"].duplicated().sum()

np.int64(10)

In [221]:
orders[orders["order_id"].duplicated(keep=False)].sort_values("order_id")

,order_id,order_date,customer_id,product_id,quantity,discount_pct,shipping_cost,sales_channel,order_status,base_price,unit_cost,unit_price,revenue,product_cost,profit
2308,O002309,2024-07-18,C01766,P0004,1,0.15,207.800080,Store,Cancelled,42415.33,30150.61,36053.03,36053.03,30150.61,5694.62
30000,O002309,2024-07-18,C01766,P0004,1,0.15,207.800080,Store,Cancelled,42415.33,30150.61,36053.03,36053.03,30150.61,5694.62
2664,O002665,2023-10-12,C01746,P0067,4,0.00,668.363776,Online,Completed,3630.57,2847.91,3630.57,14522.28,11391.64,2462.28
30004,O002665,2023-10-12,C01746,P0067,4,0.00,668.363776,Online,Completed,3630.57,2847.91,3630.57,14522.28,11391.64,2462.28
5148,O005149,2023-08-11,C00834,P0015,3,0.10,531.593509,Store,Completed,68260.89,49534.02,61434.80,184304.40,148602.06,35170.75
30006,O005149,2023-08-11,C00834,P0015,3,0.10,531.593509,Store,Completed,68260.89,49534.02,61434.80,184304.40,148602.06,35170.75
7790,O007791,2024-06-12,C00378,P0047,5,0.05,887.203084,Online,Completed,20500.61,14824.88,19475.58,97377.90,74124.40,22366.30
30007,O007791,2024-06-12,C00378,P0047,5,0.05,887.203084,Online,Completed,20500.61,14824.88,19475.58,97377.90,74124.40,22366.30
8511,O008512,2023-12-20,C00691,P0025,1,0.20,851.540355,Online,Completed,13173.89,8040.17,10539.11,10539.11,8040.17,1647.40
30005,O008512,2023-12-20,C00691,P0025,1,0.20,851.540355,Online,Completed,13173.89,8040.17,10539.11,10539.11,8040.17,1647.40


10 exact duplicate transaction records were identified.

In [222]:
orders["quantity"].value_counts().sort_index()

,count
quantity,
1,14906
2,7665
3,3582
4,2316
5,1526
20,4
25,5
50,6


In [223]:
orders["discount_pct"].describe()

,discount_pct
count,29960.000000
mean,0.091871
std,0.071049
min,0.000000
25%,0.050000
50%,0.100000
75%,0.150000
max,0.250000


In [224]:
orders["discount_pct"].value_counts(dropna=False).sort_index()

,count
discount_pct,
0.00,6074
0.05,7515
0.10,7483
0.15,4421
0.20,3030
0.25,1437
NaN,50


In [225]:
orders["order_date"].min(), orders["order_date"].max()

(Timestamp('2023-01-01 00:00:00'), Timestamp('2025-12-31 00:00:00'))

## Crosstab among the 3 tables

In [226]:
orders["customer_id"].isin(customers["customer_id"]).value_counts()

,count
customer_id,
True,30010


In [227]:
orders["product_id"].isin(products["product_id"]).value_counts()

,count
product_id,
True,30010


In [228]:
#How many orders occurred before the customer supposedly became a customer?
customer_check = orders.merge(
    customers[["customer_id", "customer_since"]],
    on="customer_id",
    how="left"
)

(customer_check["order_date"] < customer_check["customer_since"]).sum()

np.int64(6970)

This means 6,970 order records have an order date earlier than the customer's customer_since date.

That's about 23.2% of all raw order rows, which is far too large to casually dismiss as a handful of bad records.

## Let's investigate the 6,970 records

In [229]:
#First, calculate the difference between the order date and customer-since date.

customer_check["days_before_customer_since"] = (
    customer_check["customer_since"] -
    customer_check["order_date"]
).dt.days

customer_check[
    customer_check["days_before_customer_since"] > 0
]["days_before_customer_since"].describe()

,days_before_customer_since
count,6970.000000
mean,365.126255
std,258.105193
min,1.000000
25%,149.000000
50%,318.000000
75%,549.000000
max,1089.000000


In [230]:
customer_check[
    customer_check["days_before_customer_since"] > 0
].sort_values(
    "days_before_customer_since",
    ascending=False
).head(10)

,order_id,order_date,customer_id,product_id,quantity,discount_pct,shipping_cost,sales_channel,order_status,base_price,unit_cost,unit_price,revenue,product_cost,profit,customer_since,days_before_customer_since
147,O000148,2023-01-07,C02485,P0018,1,0.10,1215.992570,Store,Completed,45603.07,29794.11,41042.76,41042.76,29794.11,10032.66,2025-12-31,1089
22151,O022152,2023-01-16,C00920,P0005,1,0.05,562.691215,Online,Completed,32063.17,24460.10,30460.01,30460.01,24460.10,5437.22,2025-12-27,1076
21976,O021977,2023-01-05,C02247,P0018,2,0.00,214.609776,Online,Completed,45603.07,29794.11,45603.07,91206.14,59588.22,31403.31,2025-12-15,1075
3918,O003919,2023-01-11,C01051,P0002,2,0.05,359.717955,Online,Completed,48149.03,31099.07,45741.58,91483.16,62198.14,28925.30,2025-12-15,1069
11979,O011980,2023-01-03,C01546,P0039,4,0.20,148.031427,Online,Completed,15201.43,10476.48,12161.14,48644.56,41905.92,6590.61,2025-12-02,1064
1251,O001252,2023-01-14,C01369,P0022,1,0.15,701.567556,Store,Completed,28566.38,16306.97,24281.42,24281.42,16306.97,7272.88,2025-12-13,1064
22397,O022398,2023-01-02,C02061,P0001,2,0.05,1005.777170,Online,Completed,76307.73,44819.33,72492.34,144984.68,89638.66,54340.24,2025-11-30,1063
27281,O027282,2023-02-01,C00920,P0066,1,0.05,1440.668486,Online,Returned,1482.61,1004.14,1408.48,1408.48,1004.14,-1036.33,2025-12-27,1060
7852,O007853,2023-01-07,C01331,P0018,2,0.10,1305.123014,Store,Completed,45603.07,29794.11,41042.76,82085.52,59588.22,21192.18,2025-12-01,1059
4569,O004570,2023-02-06,C01962,P0040,3,0.00,398.050676,Online,Completed,19597.45,13150.79,19597.45,58792.35,39452.37,18941.93,2025-12-28,1056


❌ Finding - Customer master-data temporal inconsistency: 6,970 order records have transaction dates earlier than the associated customer's recorded customer_since date. The median discrepancy is 318 days, suggesting a substantive inconsistency rather than a minor timestamp issue.

"Does customer_since represent the customer's first-ever relationship with the company, or the date the current customer record/account was created?"

## Status

In [231]:
orders.groupby("order_status")[["revenue", "product_cost", "profit"]].sum()

,revenue,product_cost,profit
order_status,,,
Cancelled,3.273942e+07,2.402453e+07,8.019886e+06
Completed,9.860529e+08,7.223548e+08,2.425821e+08
Returned,7.567752e+07,5.535394e+07,1.868618e+07


In [232]:
orders.groupby("order_status").agg(
    orders=("order_id", "count"),
    revenue=("revenue", "sum"),
    profit=("profit", "sum")
)

,orders,revenue,profit
order_status,,,
Cancelled,891,3.273942e+07,8.019886e+06
Completed,27017,9.860529e+08,2.425821e+08
Returned,2102,7.567752e+07,1.868618e+07


In [233]:
orders.groupby("order_status")["profit"].mean()

,profit
order_status,
Cancelled,9000.993973
Completed,8978.868748
Returned,8889.712740


❌ Order-status and financial metrics are not aligned. Cancelled and returned orders currently carry revenue and profit values, requiring a defined treatment before calculating realized financial performance.

revenue = quantity * unit_price

In [234]:
#check the definition -> revenue = quantity * unit_price

(orders["revenue"] - orders["quantity"] * orders["unit_price"]).abs().max()

2304347.04

❌ That means the maximum difference between recorded revenue and quantity × unit_price is ₹2.3 million.

That is way too large to be rounding. So the revenue calculation is not consistently following the stated definition.

profit = revenue - product_cost - shipping_cost




In [235]:
#check the definition -> profit = revenue - product_cost - shipping_cost
(
    orders["profit"] -
    (
        orders["revenue"]
        - orders["product_cost"]
        - orders["shipping_cost"]
    )
).abs().max()

0.0049994038898830695

That's effectively zero, apart from a tiny floating-point/rounding difference.

So:

Profit calculation is correct.


In [236]:
# identify the problematic records

revenue_check = orders["revenue"] - (
    orders["quantity"] * orders["unit_price"]
)

In [237]:
orders.loc[
    revenue_check.abs().nlargest(10).index,
    [
        "order_id",
        "quantity",
        "discount_pct",
        "base_price",
        "unit_price",
        "revenue"
    ]
]

,order_id,quantity,discount_pct,base_price,unit_price,revenue
250,O000251,50,0.15,56479.09,48007.23,96014.46
14359,O014360,50,0.10,21001.14,18901.03,18901.03
22864,O022865,50,0.10,19337.72,17403.95,17403.95
26536,O026537,20,0.10,48149.03,43334.13,43334.13
18579,O018580,50,0.15,17868.64,15188.34,30376.68
10180,O010181,50,0.20,18652.90,14922.32,74611.60
25113,O025114,25,0.10,21001.14,18901.03,37802.06
25114,O025115,25,0.20,23234.15,18587.32,92936.60
24477,O024478,20,0.20,21218.15,16974.52,16974.52
21253,O021254,20,0.20,16307.91,13046.33,26092.66


The quantity field is inconsistent with the financial measures (revenue and probably product_cost) for the 15 anomalous-quantity records.

The profit formula is mathematically consistent with the stored financial fields, but those financial fields are inconsistent with the current order quantity for the anomalous records.

In [238]:
#Let's test whether product_cost has the same problem:

(
    orders["product_cost"] -
    orders["quantity"] * orders["unit_cost"]
).abs().max()

1578311.0400000003

In [239]:
#Identify problamatic record counts

revenue_mismatch = (
    orders["revenue"] -
    orders["quantity"] * orders["unit_price"]
).abs() > 0.01

cost_mismatch = (
    orders["product_cost"] -
    orders["quantity"] * orders["unit_cost"]
).abs() > 0.01

print("Revenue mismatches:", revenue_mismatch.sum())
print("Product cost mismatches:", cost_mismatch.sum())

Revenue mismatches: 15
Product cost mismatches: 15


Option A: Quantity is wrong

The original financial numbers look like they were calculated using the original quantity, and then the quantity was changed to 20/25/50.

If that's the case, we'd correct quantity back to the implied quantity.


---


Option B: Quantity is correct

The customer really ordered 20/25/50 units, but revenue and product cost weren't recalculated.

In that case, we'd need to recalculate the financial fields.


---



Because this is a portfolio project with intentionally planted issues, we know from how we generated the dataset that the quantity was deliberately changed after the financial calculations. So for our cleaning exercise, the appropriate treatment is to flag these 15 records and correct the downstream financial fields based on the current quantity, rather than deleting legitimate high-volume orders.


---



15 orders contained unusually high quantities (20–50 units). Cross-field validation showed that revenue and product cost were calculated using different quantities, so these records were flagged as inconsistent and financial measures were recalculated using the recorded quantity.


---



## Step 2: Investigate the 50 missing discounts

In [240]:
# Does a missing discount mean "no discount" (0%), or does it mean "we don't know the discount"?

orders[orders["discount_pct"].isna()][[
    "order_id",
    "order_date",
    "customer_id",
    "product_id",
    "quantity",
    "discount_pct",
    "base_price",
    "unit_price",
    "revenue",
    "order_status"
]].head(20)

,order_id,order_date,customer_id,product_id,quantity,discount_pct,base_price,unit_price,revenue,order_status
100,O000101,2023-01-12,C01970,P0014,1,NaN,16307.91,15492.51,15492.51,Completed
1012,O001013,2025-06-10,C00648,P0001,2,NaN,76307.73,68676.96,137353.92,Completed
1072,O001073,2024-08-15,C01120,P0051,2,NaN,21937.67,21937.67,43875.34,Completed
1207,O001208,2024-02-02,C02425,P0044,1,NaN,6632.18,6300.57,6300.57,Completed
1462,O001463,2023-09-16,C01800,P0007,5,NaN,33971.37,33971.37,169856.85,Completed
1628,O001629,2025-03-13,C02293,P0058,1,NaN,2016.07,1612.86,1612.86,Completed
2402,O002403,2023-10-16,C01267,P0052,1,NaN,5049.20,4796.74,4796.74,Completed
2651,O002652,2024-09-07,C02061,P0057,1,NaN,2399.62,2159.66,2159.66,Completed
2933,O002934,2025-06-13,C01918,P0011,1,NaN,76595.54,68935.99,68935.99,Completed
5206,O005207,2025-09-25,C00784,P0052,2,NaN,5049.20,5049.20,10098.40,Completed


Calculating discounts for missing fields from unit_price and base_price

In [241]:
missing_discount = orders["discount_pct"].isna()

implied_discount = (
    1 - orders.loc[missing_discount, "unit_price"]
    / orders.loc[missing_discount, "base_price"]
)

implied_discount.describe()

,0
count,50.000000
mean,0.081000
std,0.073464
min,0.000000
25%,0.012500
50%,0.050000
75%,0.100001
max,0.250001


In [242]:
implied_discount.round(2).value_counts().sort_index()

,count
0.00,13
0.05,15
0.10,10
0.15,4
0.20,6
0.25,2


In [243]:
implied_discount = (
    1 - orders["unit_price"] / orders["base_price"]
)

orders.loc[orders["discount_pct"].isna(), "implied_discount"] = implied_discount[
    orders["discount_pct"].isna()
]

(
    orders.loc[orders["discount_pct"].isna(), "implied_discount"]
    .round(2)
    .value_counts()
    .sort_index()
)

,count
implied_discount,
0.00,13
0.05,15
0.10,10
0.15,4
0.20,6
0.25,2


### Next issue: missing unit_cost in Products

In [244]:
products[products["unit_cost"].isna()]

,product_id,product_name,category,subcategory,unit_cost,base_price
10,P0011,Mobile Phones Product 5,Electronics,Mobile Phones,NaN,76595.54
15,P0016,Accessories Product 4,Electronics,Accessories,NaN,77138.70
31,P0032,Storage Product 2,Furniture,Storage,NaN,24167.24
69,P0070,Writing Product 4,Office Supplies,Writing,NaN,421.13


In [245]:
products.groupby(["category","subcategory"])[["unit_cost", "base_price"]].agg(
    ["count", "mean", "median", "min", "max"]
)

unit_cost                                     \
                                  count          mean     median       min   
category        subcategory                                                  
Electronics     Accessories           5  30567.754000  29794.110   8998.28   
                Laptops               6  31355.705000  29868.990  24460.10   
                Mobile Phones         5  29056.380000  28887.300  19746.44   
Furniture       Chairs                6   9166.576667   8783.250   3002.06   
                Desks                 6  11545.455000  11954.730   5837.13   
                Storage               5  13895.856000  17504.260   4830.10   
Home Appliances Cleaning              6   8664.251667   8794.980   2040.25   
                Climate               6   7457.046667   5416.605   2285.05   
                Kitchen               6   9378.648333  10584.800   1641.83   
Office Supplies Paper                 6   1651.711667   1471.955   1004.14   
                Stationery            6   2016.040000   1926.635   1477.21   
                Writing               5   1519.398000    572.900    520.17   

                                        base_price                           \
                                    max      count          mean     median   
category        subcategory                                                   
Electronics     Accessories    49534.02          6  48916.888333  48707.030   
                Laptops        44819.33          6  48716.256667  46681.140   
                Mobile Phones  39856.38          6  49830.943333  48393.725   
Furniture       Chairs         16306.97          6  14244.706667  12979.945   
                Desks          16854.64          6  17416.558333  17094.430   
                Storage        21199.12          6  20750.063333  24871.220   
Home Appliances Cleaning       14824.88          6  13156.161667  13566.395   
                Climate        15735.09          6  11156.040000   8824.590   
                Kitchen        13517.06          6  14968.300000  16121.650   
Office Supplies Paper           3036.56          6   2474.610000   2387.770   
                Stationery      2929.93          6   2982.518333   2818.460   
                Writing         3126.80          6   1792.733333    803.180   

                                                   
                                    min       max  
category        subcategory                        
Electronics     Accessories    16307.91  77138.70  
                Laptops        32063.17  76307.73  
                Mobile Phones  33971.37  76595.54  
Furniture       Chairs          4047.59  28566.38  
                Desks           9763.04  26372.42  
                Storage         6233.16  28469.37  
Home Appliances Cleaning        2648.40  23120.56  
                Climate         3620.60  21937.67  
                Kitchen         2788.85  23234.15  
Office Supplies Paper           1482.61   3877.55  
                Stationery      2016.07   3944.40  
                Writing          421.13   4370.25

In [246]:
products[
    products["subcategory"].isin(
        ["Accessories", "Mobile Phones", "Storage", "Writing"]
    )
][[
    "product_id",
    "product_name",
    "category",
    "subcategory",
    "unit_cost",
    "base_price"
]].sort_values(["subcategory", "product_id"])

,product_id,product_name,category,subcategory,unit_cost,base_price
12,P0013,Accessories Product 1,Electronics,Accessories,39926.78,51810.99
13,P0014,Accessories Product 2,Electronics,Accessories,8998.28,16307.91
14,P0015,Accessories Product 3,Electronics,Accessories,49534.02,68260.89
15,P0016,Accessories Product 4,Electronics,Accessories,NaN,77138.70
16,P0017,Accessories Product 5,Electronics,Accessories,24585.58,34379.77
17,P0018,Accessories Product 6,Electronics,Accessories,29794.11,45603.07
6,P0007,Mobile Phones Product 1,Electronics,Mobile Phones,23910.30,33971.37
7,P0008,Mobile Phones Product 2,Electronics,Mobile Phones,28887.30,41776.53
8,P0009,Mobile Phones Product 3,Electronics,Mobile Phones,39856.38,55010.92
9,P0010,Mobile Phones Product 4,Electronics,Mobile Phones,32881.48,56479.09


For the products where unit_cost is known, calculate the markup ratio:

In [247]:
products["cost_to_price_ratio"] = (
    products["unit_cost"] / products["base_price"]
)

products[products["unit_cost"].notna()][[
    "product_id",
    "subcategory",
    "unit_cost",
    "base_price",
    "cost_to_price_ratio"
]].sort_values("subcategory")

,product_id,subcategory,unit_cost,base_price,cost_to_price_ratio
17,P0018,Accessories,29794.11,45603.07,0.653336
16,P0017,Accessories,24585.58,34379.77,0.715118
14,P0015,Accessories,49534.02,68260.89,0.725657
13,P0014,Accessories,8998.28,16307.91,0.551774
12,P0013,Accessories,39926.78,51810.99,0.770624
...,...,...,...,...,...
66,P0067,Writing,2847.91,3630.57,0.784425
67,P0068,Writing,520.17,831.51,0.625573
68,P0069,Writing,572.90,728.09,0.786853
70,P0071,Writing,3126.80,4370.25,0.715474


Because we have only 4 missing values, we don't need anything fancy. We can use the relationship between base_price and unit_cost from the other 68 products.

Specifically, let's see whether base_price actually predicts unit_cost reasonably well.

In [248]:
products[products["unit_cost"].notna()][
    ["unit_cost", "base_price"]
].corr()

,unit_cost,base_price
unit_cost,1.000000,0.989814
base_price,0.989814,1.000000


What 0.9898 tells us

It tells us:

Products with higher selling prices tend to have proportionally higher company costs.

It does not by itself tell us the exact cost for a missing product.

For that, let's fit a simple linear relationship:

unit_cost=a+b×base_price

using the 68 products where cost is known.

Then we'll use that relationship to estimate the four missing costs.

In [249]:
# Running Linear Regression

from sklearn.linear_model import LinearRegression

known = products[products["unit_cost"].notna()]

X = known[["base_price"]]
y = known["unit_cost"]

model = LinearRegression()
model.fit(X, y)

print("Intercept:", model.intercept_)
print("Slope:", model.coef_[0])
print("R²:", model.score(X, y))

Intercept: 65.08009279644284
Slope: 0.6600847138430472
R²: 0.9797312678102381


unit_cost=65.08+0.6601×base_price

and:

R² = 0.9797

That means the linear relationship explains about 98% of the variation in unit cost in the 68 products with known costs. That's excellent.

Before filling the four missing values, let's calculate what the model predicts and inspect them.

In [250]:
missing = products["unit_cost"].isna()

products.loc[missing, "estimated_unit_cost"] = model.predict(
    products.loc[missing, ["base_price"]]
)

products.loc[missing, [
    "product_id",
    "product_name",
    "subcategory",
    "base_price",
    "estimated_unit_cost"
]]

,product_id,product_name,subcategory,base_price,estimated_unit_cost
10,P0011,Mobile Phones Product 5,Mobile Phones,76595.54,50624.625195
15,P0016,Accessories Product 4,Accessories,77138.70,50983.156809
31,P0032,Storage Product 2,Storage,24167.24,16017.505793
69,P0070,Writing Product 4,Writing,421.13,343.061568


Four missing product costs were imputed using a linear regression of unit_cost on base_price, trained on the 68 products with known costs. The model had an R² of 0.980, indicating a strong relationship between selling price and product cost.

In [251]:
known["predicted_unit_cost"] = model.predict(
    known[["base_price"]]
)

known["prediction_error"] = (
    known["unit_cost"] - known["predicted_unit_cost"]
)

known["prediction_error"].describe()

/tmp/ipykernel_2839/3221582700.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  known["predicted_unit_cost"] = model.predict(
/tmp/ipykernel_2839/3221582700.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  known["prediction_error"] = (


,prediction_error
count,6.800000e+01
mean,1.387648e-12
std,1.718373e+03
min,-5.615316e+03
25%,-6.338744e+02
50%,-4.346251e+01
75%,4.871546e+02
max,5.662057e+03


4 missing unit_cost values → impute using the linear regression based on base_price.

The reasoning is:

1. Only 4 of 72 products are missing cost.
2. base_price and unit_cost have an extremely strong relationship, correlation 0.9898.
3. The regression has R² = 0.9797.
4. The resulting estimates are economically sensible.
5. A generic median would produce particularly questionable results for the Writing product.

## Next: the 30 missing customer cities

In [252]:
customers[customers["city"].isna()][[
    "customer_id",
    "customer_name",
    "customer_segment",
    "city",
    "state",
    "customer_since"
]]

,customer_id,customer_name,customer_segment,city,state,customer_since
55,C00056,Customer_56,Enterprise,NaN,Gujarat,2022-02-26
129,C00130,Customer_130,Consumer,NaN,Kerala,2025-07-17
182,C00183,Customer_183,Enterprise,NaN,Kerala,2022-07-27
219,C00220,Customer_220,Small Business,NaN,Tamil Nadu,2025-08-22
350,C00351,Customer_351,Consumer,NaN,Maharashtra,2020-10-10
415,C00416,Customer_416,Consumer,NaN,Gujarat,2021-07-20
461,C00462,Customer_462,Small Business,NaN,Maharashtra,2023-04-21
498,C00499,Customer_499,Enterprise,NaN,Maharashtra,2021-06-30
555,C00556,Customer_556,Consumer,NaN,Telangana,2022-07-08
766,C00767,Customer_767,Small Business,NaN,Tamil Nadu,2024-09-25


In [253]:
customers.groupby("state")["city"].unique()

,city
state,
Delhi,[Delhi]
Gujarat,"[Ahmedabad, nan]"
Karnataka,"[Bengaluru, nan]"
Kerala,"[Kochi, nan]"
Maharashtra,"[Mumbai, Pune, nan]"
Rajasthan,[Jaipur]
Tamil Nadu,"[Chennai, nan]"
Telangana,"[Hyderabad, nan]"
West Bengal,"[Kolkata, nan]"


Keep the city as missing, or replace it with "Unknown" only in the analysis layer.

In [254]:
customer_check["days_before_customer_since"] = (
    customer_check["customer_since"] - customer_check["order_date"]
).dt.days

customer_check[
    customer_check["order_date"] < customer_check["customer_since"]
][[
    "customer_id",
    "customer_since",
    "order_date",
    "days_before_customer_since"
]].sort_values(
    "days_before_customer_since",
    ascending=False
).head(20)

,customer_id,customer_since,order_date,days_before_customer_since
147,C02485,2025-12-31,2023-01-07,1089
22151,C00920,2025-12-27,2023-01-16,1076
21976,C02247,2025-12-15,2023-01-05,1075
3918,C01051,2025-12-15,2023-01-11,1069
11979,C01546,2025-12-02,2023-01-03,1064
1251,C01369,2025-12-13,2023-01-14,1064
22397,C02061,2025-11-30,2023-01-02,1063
27281,C00920,2025-12-27,2023-02-01,1060
7852,C01331,2025-12-01,2023-01-07,1059
4569,C01962,2025-12-28,2023-02-06,1056


In [255]:
date_issue = customer_check[
    customer_check["order_date"] < customer_check["customer_since"]
]

date_issue["customer_id"].nunique()

1097

In [256]:
date_issue.groupby("customer_id").size().sort_values(
    ascending=False
).head(15)

,0
customer_id,
C01324,20
C00207,20
C00099,20
C01779,19
C01786,19
C02306,18
C02061,18
C00614,17
C01069,17


1. 1,097 / 2,500 customers affected, about 43.9%
2. 6,970 / 30,010 orders affected, about 23.2%
3. The worst customers have ~20 affected orders, so this isn't just a few customers dominating the problem.
4. Some customer_since dates are in late 2025 while their orders go back to early 2023.

We don't have enough evidence to know whether:

customer_since actually means "first-ever customer relationship", in which case many records are wrong, or
it means something like "current account/profile creation date", in which case the historical orders could legitimately predate it.

Therefore, the correct treatment is:

Flag the inconsistency and retain the records.

We already saw that cancelled and returned orders currently have positive revenue/profit. We need to decide what revenue means in this project.

In [257]:
orders.groupby("order_status").agg(
    orders=("order_id", "count"),
    revenue=("revenue", "sum"),
    profit=("profit", "sum"),
    avg_revenue=("revenue", "mean")
)

,orders,revenue,profit,avg_revenue
order_status,,,,
Cancelled,891,3.273942e+07,8.019886e+06,36744.574815
Completed,27017,9.860529e+08,2.425821e+08,36497.496267
Returned,2102,7.567752e+07,1.868618e+07,36002.623834


Realized revenue = revenue from Completed orders only.

And keep the original revenue as the transaction-level gross order value.

# Phase 2: Create the cleaned datasets

In [258]:
orders_clean = orders.copy()
customers_clean = customers.copy()
products_clean = products.copy()

### Step 1: Remove the exact duplicate orders

In [259]:
orders_clean = orders_clean.drop_duplicates()

print("Rows after removing duplicates:", len(orders_clean))
print("Unique order IDs:", orders_clean["order_id"].nunique())
print("Duplicate rows:", orders_clean.duplicated().sum())

Rows after removing duplicates: 30000
Unique order IDs: 30000
Duplicate rows: 0


### Step 2: Impute the missing discounts

In [260]:
missing_discount = orders_clean["discount_pct"].isna()

orders_clean.loc[missing_discount, "discount_pct"] = (
    1 -
    orders_clean.loc[missing_discount, "unit_price"]
    / orders_clean.loc[missing_discount, "base_price"]
)

print("Missing discounts:", orders_clean["discount_pct"].isna().sum())

Missing discounts: 0


In [261]:
orders_clean["discount_pct"].value_counts().sort_index()

,count
discount_pct,
0.000000,6086
0.049999,1
0.050000,1
0.050000,1
0.050000,1
0.050000,1
0.050000,1
0.050000,1
0.050000,1


In [262]:
orders_clean["discount_pct"] = orders_clean["discount_pct"].round(2)

orders_clean["discount_pct"].value_counts().sort_index()

,count
discount_pct,
0.00,6086
0.05,7529
0.10,7489
0.15,4422
0.20,3035
0.25,1439


### Step 3: Fix the 15 quantity-related financial inconsistencies

quantity × unit_price ≠ revenue

quantity × unit_cost ≠ product_cost

In [263]:
revenue_mismatch = (
    orders_clean["revenue"] -
    orders_clean["quantity"] * orders_clean["unit_price"]
).abs() > 0.01

cost_mismatch = (
    orders_clean["product_cost"] -
    orders_clean["quantity"] * orders_clean["unit_cost"]
).abs() > 0.01

print("Revenue mismatches:", revenue_mismatch.sum())
print("Product cost mismatches:", cost_mismatch.sum())

Revenue mismatches: 15
Product cost mismatches: 15


In [264]:
orders_clean["revenue"] = (
    orders_clean["quantity"] * orders_clean["unit_price"]
)

orders_clean["product_cost"] = (
    orders_clean["quantity"] * orders_clean["unit_cost"]
)

orders_clean["profit"] = (
    orders_clean["revenue"]
    - orders_clean["product_cost"]
    - orders_clean["shipping_cost"]
)

In [265]:
revenue_check = (
    orders_clean["revenue"]
    - orders_clean["quantity"] * orders_clean["unit_price"]
).abs()

cost_check = (
    orders_clean["product_cost"]
    - orders_clean["quantity"] * orders_clean["unit_cost"]
).abs()

profit_check = (
    orders_clean["profit"]
    - (
        orders_clean["revenue"]
        - orders_clean["product_cost"]
        - orders_clean["shipping_cost"]
    )
).abs()

print("Revenue mismatches:", (revenue_check > 0.01).sum())
print("Product cost mismatches:", (cost_check > 0.01).sum())
print("Profit mismatches:", (profit_check > 0.01).sum())

Revenue mismatches: 0
Product cost mismatches: 0
Profit mismatches: 0


### Step 4: Handle the 4 missing product costs

In [266]:
from sklearn.linear_model import LinearRegression

known_cost = products_clean["unit_cost"].notna()

X = products_clean.loc[known_cost, ["base_price"]]
y = products_clean.loc[known_cost, "unit_cost"]

model = LinearRegression()
model.fit(X, y)

print("Intercept:", model.intercept_)
print("Slope:", model.coef_[0])
print("R²:", model.score(X, y))

Intercept: 65.08009279644284
Slope: 0.6600847138430472
R²: 0.9797312678102381


In [267]:
missing_cost = products_clean["unit_cost"].isna()

products_clean.loc[missing_cost, "unit_cost"] = (
    model.predict(
        products_clean.loc[missing_cost, ["base_price"]]
    )
)

In [268]:
products_clean.loc[
    missing_cost,
    ["product_id", "product_name", "category", "base_price", "unit_cost"]
]

,product_id,product_name,category,base_price,unit_cost
10,P0011,Mobile Phones Product 5,Electronics,76595.54,50624.625195
15,P0016,Accessories Product 4,Electronics,77138.70,50983.156809
31,P0032,Storage Product 2,Furniture,24167.24,16017.505793
69,P0070,Writing Product 4,Office Supplies,421.13,343.061568


"Four missing unit-cost values were imputed using a linear regression model based on base price. The model was trained on 68 products with known costs and achieved an in-sample R² of 0.98. Imputed values are treated as estimates rather than recovered source values."

In [269]:
products_clean["unit_cost"].isna().sum()

np.int64(0)

In [270]:
(products_clean["unit_cost"] > products_clean["base_price"]).sum()

np.int64(0)

In [271]:
products_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72 entries, 0 to 71
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   product_id           72 non-null     object 
 1   product_name         72 non-null     object 
 2   category             72 non-null     object 
 3   subcategory          72 non-null     object 
 4   unit_cost            72 non-null     float64
 5   base_price           72 non-null     float64
 6   cost_to_price_ratio  68 non-null     float64
 7   estimated_unit_cost  4 non-null      float64
dtypes: float64(4), object(4)
memory usage: 4.6+ KB


### Step 5A: Create the validation flag

In [272]:
# order_date < customer_since for 6,970 orders, involving 1,097 customers.

Flag the inconsistency, preserve both original dates, and exclude/handle flagged records explicitly when an analysis depends on customer tenure.

In [273]:
orders_clean = orders_clean.merge(
    customers_clean[["customer_id", "customer_since"]],
    on="customer_id",
    how="left"
)

In [274]:
orders_clean["customer_date_issue"] = (
    orders_clean["order_date"] < orders_clean["customer_since"]
)


### Step 5B: How widespread is the problem?

In [275]:
#Check 1: How many problematic orders?

orders_clean["customer_date_issue"].value_counts()

,count
customer_date_issue,
False,23032
True,6968


In [276]:
#Check 2: How many different customers are affected?

orders_clean.loc[
    orders_clean["customer_date_issue"],
    "customer_id"
].nunique()

1097

### Step 5C: How severe is the problem?

In [277]:
#How many days before customer_since the problematic orders occurred ?

date_issue = orders_clean[
    orders_clean["customer_date_issue"]
].copy()

date_issue["days_before_customer_since"] = (
    date_issue["customer_since"] - date_issue["order_date"]
).dt.days


In [278]:
date_issue["days_before_customer_since"].describe()

,days_before_customer_since
count,6968.000000
mean,365.175804
std,258.124751
min,1.000000
25%,149.000000
50%,318.000000
75%,549.250000
max,1089.000000


## Validation

### Step 6A: Check the shapes

In [279]:
print("Customers:", customers_clean.shape)
print("Products:", products_clean.shape)
print("Orders:", orders_clean.shape)

Customers: (2500, 6)
Products: (72, 8)
Orders: (30000, 18)


### Step 6B: Check missing values

In [280]:
print("Customers missing:")
print(customers_clean.isna().sum())

print("\nProducts missing:")
print(products_clean.isna().sum())

print("\nOrders missing:")
print(orders_clean.isna().sum())

Customers missing:
customer_id          0
customer_name        0
customer_segment     0
city                30
state                0
customer_since       0
dtype: int64

Products missing:
product_id              0
product_name            0
category                0
subcategory             0
unit_cost               0
base_price              0
cost_to_price_ratio     4
estimated_unit_cost    68
dtype: int64

Orders missing:
order_id                   0
order_date                 0
customer_id                0
product_id                 0
quantity                   0
discount_pct               0
shipping_cost              0
sales_channel              0
order_status               0
base_price                 0
unit_cost                  0
unit_price                 0
revenue                    0
product_cost               0
profit                     0
implied_discount       29950
customer_since             0
customer_date_issue        0
dtype: int64


### Step 6B.1: Remove the helper columns

In [281]:
orders_clean = orders_clean.drop(columns=["implied_discount"])

products_clean = products_clean.drop(
    columns=["cost_to_price_ratio", "estimated_unit_cost"]
)

print("Customers missing:")
print(customers_clean.isna().sum())

print("\nProducts missing:")
print(products_clean.isna().sum())

print("\nOrders missing:")
print(orders_clean.isna().sum())

Customers missing:
customer_id          0
customer_name        0
customer_segment     0
city                30
state                0
customer_since       0
dtype: int64

Products missing:
product_id      0
product_name    0
category        0
subcategory     0
unit_cost       0
base_price      0
dtype: int64

Orders missing:
order_id               0
order_date             0
customer_id            0
product_id             0
quantity               0
discount_pct           0
shipping_cost          0
sales_channel          0
order_status           0
base_price             0
unit_cost              0
unit_price             0
revenue                0
product_cost           0
profit                 0
customer_since         0
customer_date_issue    0
dtype: int64


### Step 6C: Check duplicate IDs

In [282]:
print("Duplicate customer IDs:",
      customers_clean["customer_id"].duplicated().sum())

print("Duplicate product IDs:",
      products_clean["product_id"].duplicated().sum())

print("Duplicate order IDs:",
      orders_clean["order_id"].duplicated().sum())

Duplicate customer IDs: 0
Duplicate product IDs: 0
Duplicate order IDs: 0


### 6D: Referential Integrity

In [283]:
# Does every order actually point to a customer and product that exists in our master tables?

print(
    "Invalid customer references:",
    (~orders_clean["customer_id"].isin(customers_clean["customer_id"])).sum()
)

print(
    "Invalid product references:",
    (~orders_clean["product_id"].isin(products_clean["product_id"])).sum()
)

Invalid customer references: 0
Invalid product references: 0


### Step 6E: Validate the financial calculations

In [284]:
revenue_check = (
    orders_clean["revenue"]
    - orders_clean["quantity"] * orders_clean["unit_price"]
).abs()

cost_check = (
    orders_clean["product_cost"]
    - orders_clean["quantity"] * orders_clean["unit_cost"]
).abs()

profit_check = (
    orders_clean["profit"]
    - (
        orders_clean["revenue"]
        - orders_clean["product_cost"]
        - orders_clean["shipping_cost"]
    )
).abs()

print("Revenue mismatches:", (revenue_check > 0.01).sum())
print("Product cost mismatches:", (cost_check > 0.01).sum())
print("Profit mismatches:", (profit_check > 0.01).sum())

Revenue mismatches: 0
Product cost mismatches: 0
Profit mismatches: 0


### Step 6F: Final sanity check

In [286]:
#Check that the cleaned data still has sensible values

print("Negative quantities:",
      (orders_clean["quantity"] < 0).sum())

print("Negative discounts:",
      (orders_clean["discount_pct"] < 0).sum())

print("Discounts above 25%:",
      (orders_clean["discount_pct"] > 0.25).sum())

print("Negative revenue:",
      (orders_clean["revenue"] < 0).sum())

print("Negative product cost:",
      (orders_clean["product_cost"] < 0).sum())

#Check that basic business-domain values remain sensible after all our transformations

print("Order statuses:")
print(orders_clean["order_status"].value_counts())

print("\nSales channels:")
print(orders_clean["sales_channel"].value_counts())

Negative quantities: 0
Negative discounts: 0
Discounts above 25%: 0
Negative revenue: 0
Negative product cost: 0
Order statuses:
order_status
Completed    27008
Returned      2102
Cancelled      890
Name: count, dtype: int64

Sales channels:
sales_channel
Online    21101
Store      8899
Name: count, dtype: int64


## Save Data to CSVs

In [287]:
orders_clean.to_csv(
    folder_path+"orders_clean.csv",
    index=False
)

customers_clean.to_csv(
    folder_path+"customers_clean.csv",
    index=False
)

products_clean.to_csv(
    folder_path+"products_clean.csv",
    index=False
)
